# ARS 2026 revision run

Every analysis the reviewers asked for. Settings: Accelerator `None`, Internet `On`.

Run cells 1-5, then the quick test. If it is clean, use `Save Version -> Save & Run All` for the full run (about 3 hours).

In [ ]:
!pip install -q ucimlrepo kagglehub xgboost imbalanced-learn statsmodels

In [ ]:
%%writefile dupaudit.py
"""dupaudit: audit exact and near-duplicate records in tabular ML datasets.

Usage:
    python dupaudit.py data.csv --target TARGET [--positive LABEL] [--eps 0.1 0.25] [--out report.json]
"""
import argparse
import json

import numpy as np
import pandas as pd
from scipy.sparse.csgraph import connected_components
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ID_LIKE = {"id", "patient_id", "patientid", "index", "unnamed: 0"}
MISSING_TOKENS = {"?", "nan", "none", "", "na", "n/a"}


def clean_frame(df, target):
    """Strip names/strings, drop ID-like columns, coerce numeric-looking text, bool->int."""
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    target = str(target).strip()
    dropped = [c for c in df.columns if c.lower() in ID_LIKE and c != target]
    df = df.drop(columns=dropped)
    for c in df.columns:
        if df[c].dtype == bool:
            df[c] = df[c].astype(int)
        elif pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c]):
            s = df[c].astype(str).str.strip()
            s = s.where(~s.str.lower().isin(MISSING_TOKENS), np.nan)
            num = pd.to_numeric(s, errors="coerce")
            if s.notna().sum() > 0 and num.notna().sum() >= 0.9 * s.notna().sum():
                df[c] = num
            else:
                df[c] = s
    return df, dropped


def binarize_target(y, positive=None):
    y = pd.Series(y).reset_index(drop=True)
    if pd.api.types.is_object_dtype(y) or pd.api.types.is_string_dtype(y):
        y = y.astype(object).where(y.isna(), y.astype(str).str.strip())
    vals = sorted(pd.unique(y.dropna()), key=str)
    if positive is not None:
        yb = (y == positive).astype(int)
        if yb.nunique() < 2:
            raise ValueError(f"positive={positive!r} not found in target values {vals[:10]}")
        return yb, {"positive": str(positive)}
    if len(vals) == 2:
        return (y == vals[1]).astype(int), {"positive": str(vals[1])}
    yn = pd.to_numeric(y, errors="coerce")
    if yn.notna().all():
        return (yn > 0).astype(int), {"positive": ">0"}
    raise ValueError(f"Cannot binarize target with values {vals[:10]}")


def make_preprocessor(X):
    num = X.select_dtypes(include=[np.number]).columns.tolist()
    cat = [c for c in X.columns if c not in num]
    steps = []
    if num:
        steps.append(("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                       ("sc", StandardScaler())]), num))
    if cat:
        steps.append(("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                       ("oh", OneHotEncoder(handle_unknown="ignore",
                                                            sparse_output=False))]), cat))
    return ColumnTransformer(steps)


def row_keys(X):
    """Stable hash per feature vector (NaN-aware)."""
    return pd.util.hash_pandas_object(X.astype(str), index=False).values


def near_duplicate_rows(X, eps):
    """Rows absorbed into clusters whose members lie within `eps` (standardized Euclidean)."""
    Z = make_preprocessor(X).fit_transform(X)
    A = NearestNeighbors(radius=eps).fit(Z).radius_neighbors_graph(Z, mode="connectivity")
    n_clusters, _ = connected_components(A, directed=False)
    return int(len(X) - n_clusters)


def audit(df, target, eps_list=(0.1, 0.25)):
    """df must already be cleaned; target column must be binary 0/1."""
    X = df.drop(columns=[target])
    y = df[target].reset_index(drop=True)
    n = len(df)
    keys = row_keys(X)
    labels_per_vec = pd.Series(y.values).groupby(keys).nunique()
    out = {
        "n_rows": int(n),
        "n_features": int(X.shape[1]),
        "positive_rate": round(float(y.mean()), 4),
        "exact_dup_rows_feat_label": int(df.duplicated(keep="first").sum()),
        "exact_dup_rows_feat_only": int(X.duplicated(keep="first").sum()),
        "unique_feature_vectors": int(len(labels_per_vec)),
        "conflicting_label_groups": int((labels_per_vec > 1).sum()),
    }
    out["exact_dup_pct"] = round(100 * out["exact_dup_rows_feat_label"] / n, 2)
    ded = df.drop_duplicates()
    out["n_rows_after_dedup"] = int(len(ded))
    out["positive_rate_after_dedup"] = round(float(ded[target].mean()), 4)
    for eps in eps_list:
        out[f"near_dup_rows_eps{eps}"] = near_duplicate_rows(X, eps)
    return out


def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("csv")
    ap.add_argument("--target", required=True)
    ap.add_argument("--positive", default=None)
    ap.add_argument("--eps", type=float, nargs="*", default=[0.1, 0.25])
    ap.add_argument("--out", default=None)
    a = ap.parse_args()
    raw = pd.read_csv(a.csv)
    df, dropped = clean_frame(raw, a.target)
    tgt = str(a.target).strip()
    df[tgt], _ = binarize_target(df[tgt], a.positive)
    df = df.dropna(subset=[tgt]).reset_index(drop=True)
    rep = audit(df, tgt, a.eps)
    rep["dropped_id_columns"] = dropped
    for k, v in rep.items():
        print(f"{k:32s} {v}")
    if a.out:
        with open(a.out, "w") as f:
            json.dump(rep, f, indent=2)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile audit_v2.py
"""Revision-stage audit additions (Reviewer 1 points 2 and 7)."""
import hashlib
import numpy as np
import pandas as pd
from dupaudit import make_preprocessor, row_keys


def frame_checksum(df):
    """Stable sha256 of the cleaned frame, so a reader can confirm the same file."""
    b = pd.util.hash_pandas_object(df.astype(str), index=False).values.tobytes()
    return hashlib.sha256(b).hexdigest()[:16]


def feature_space_profile(X):
    """R1.7: can identical feature vectors arise by coincidence rather than from one patient?

    Reports the cardinality of each feature and the size of the discrete grid they span.
    A grid far larger than n makes coincidental identity implausible; a grid near or below n
    makes it likely, which is the patient-level versus record-level distinction the reviewer
    asks for.
    """
    card = {c: int(X[c].nunique(dropna=False)) for c in X.columns}
    n_cont = sum(1 for c in X.columns
                 if pd.api.types.is_numeric_dtype(X[c]) and X[c].nunique() > 20)
    log10_grid = float(np.sum([np.log10(max(v, 1)) for v in card.values()]))
    return dict(cardinalities=card,
                min_cardinality=min(card.values()),
                median_cardinality=float(np.median(list(card.values()))),
                n_high_cardinality_features=n_cont,
                log10_grid_size=round(log10_grid, 2),
                log10_n=round(float(np.log10(len(X))), 2),
                coincidence_plausible=bool(log10_grid < np.log10(len(X)) + 2))


def residual_overlap_under_dedup(df, target="y"):
    """R1.2: deduplicating on (features, label) leaves conflicting-label records behind.

    Returns the number of rows that remain duplicated at the FEATURE level after an
    exact (features+label) deduplication, which is the residual leakage channel.
    """
    X = df.drop(columns=[target])
    ded = df.drop_duplicates()                      # features + label
    Xd = ded.drop(columns=[target])
    kd = row_keys(Xd)
    vc = pd.Series(kd).value_counts()
    repeated = vc[vc > 1]
    return dict(rows_after_dedup=int(len(ded)),
                feature_vectors_still_repeated=int(len(repeated)),
                rows_in_repeated_groups=int(repeated.sum()),
                residual_pct=round(100 * float(repeated.sum()) / max(len(ded), 1), 2))


In [ ]:
%%writefile experiments_v2.py
"""ARS 2026 revision run. Extends the original experiment to answer the reviewers.

Protocols (all stratified five-fold CV, repeated over seeds):
  P0_naive  : StratifiedKFold on the raw file (common practice).
  P1_group  : StratifiedGroupKFold grouped on the FEATURE vector, so every copy of a
              record -- including copies that carry conflicting labels -- stays on one
              side. Same rows as P0. Primary comparator. (R1.2)
  P2_dedup  : drop exact duplicates on (features + label), then StratifiedKFold.
              Leaves conflicting-label records behind, which is measured. (R1.2)
  P2b_dedup_feat : drop duplicates on FEATURES only, then StratifiedKFold. Removes the
              residual channel P2 leaves open. (R1.2)

Per-fold bookkeeping (R1.1): fold sizes and class balance in train and test are recorded
for every fold, so the fold-geometry confound can be measured rather than assumed.

Oversampling (R1.5), three settings on deduplicated data:
  smote_in_fold        : SMOTE inside training folds, real test rows.      (correct)
  smote_pre_resampled  : SMOTE before the split, test fold as resampled.   (what papers report)
  smote_pre_realtest   : SMOTE before the split, test fold filtered to REAL rows only.
                         Isolates leakage from the change in test-set composition.
"""
import glob, hashlib, json, os, platform, time, warnings

import matplotlib; matplotlib.use("Agg")
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score)
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from dupaudit import audit, binarize_target, clean_frame, make_preprocessor, row_keys
from audit_v2 import feature_space_profile, frame_checksum, residual_overlap_under_dedup

warnings.filterwarnings("ignore")
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    from sklearn.ensemble import HistGradientBoostingClassifier
    HAS_XGB = False

DATASETS = {
    "heart_kaggle_1025": dict(src="kaggle", slug="johnsmith88/heart-disease-dataset",
                              file="heart.csv", target="target"),
    "heart_failure_918": dict(src="kaggle", slug="fedesoriano/heart-failure-prediction",
                              file="heart.csv", target="HeartDisease"),
    "stroke_kaggle": dict(src="kaggle", slug="fedesoriano/stroke-prediction-dataset",
                          file="healthcare-dataset-stroke-data.csv", target="stroke"),
    "cleveland_uci": dict(src="uci", id=45),
    "ckd_uci": dict(src="uci", id=336, positive="ckd"),
    "ilpd_uci": dict(src="uci", id=225, positive=1),
    "early_diabetes_uci": dict(src="uci", id=529, positive="Positive"),
    "maternal_risk_uci": dict(src="uci", id=863, positive="high risk"),
    "lung_cancer_kaggle": dict(src="kaggle", slug="mysarahmadbhat/lung-cancer",
                               file="survey lung cancer.csv", target="LUNG_CANCER",
                               positive="YES"),
}
METRICS = ["acc", "bacc", "f1", "auc"]


# ------------------------------------------------------------------ loading
def load_raw(spec):
    if spec["src"] == "frame":
        return spec["df"].copy(), spec["target"]
    if spec["src"] == "kaggle":
        import kagglehub
        path = kagglehub.dataset_download(spec["slug"])
        f = os.path.join(path, spec["file"])
        if not os.path.exists(f):
            f = sorted(glob.glob(os.path.join(path, "**", "*.csv"), recursive=True))[0]
        return pd.read_csv(f), spec["target"]
    if spec["src"] == "uci":
        from ucimlrepo import fetch_ucirepo
        ds = fetch_ucirepo(id=spec["id"])
        X = ds.data.features.copy()
        t = ds.data.targets.iloc[:, 0].rename("__target__")
        return pd.concat([X.reset_index(drop=True), t.reset_index(drop=True)], axis=1), "__target__"
    raise ValueError(spec["src"])


def prepare(spec):
    raw, target = load_raw(spec)
    df, dropped = clean_frame(raw, target)
    target = str(target).strip()
    df = df.dropna(subset=[target]).reset_index(drop=True)
    df["y"], info = binarize_target(df[target], spec.get("positive"))
    if target != "y":
        df = df.drop(columns=[target])
    return df, dropped, info


# ------------------------------------------------------------------ models
def get_models(seed):
    m = {"LR": LogisticRegression(max_iter=3000),
         "RF": RandomForestClassifier(n_estimators=200, random_state=seed, n_jobs=-1),
         "kNN": KNeighborsClassifier(n_neighbors=5),
         "SVM": SVC(kernel="rbf", gamma="scale", random_state=seed)}
    m["XGB"] = (XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                              subsample=0.9, eval_metric="logloss", random_state=seed,
                              n_jobs=-1, verbosity=0) if HAS_XGB
                else HistGradientBoostingClassifier(random_state=seed))
    return m


def scores(model, X, y):
    pred = model.predict(X)
    try:
        s = model.predict_proba(X)[:, 1]
    except (AttributeError, NotImplementedError):
        s = model.decision_function(X)
    auc = roc_auc_score(y, s) if len(np.unique(y)) == 2 else np.nan
    return dict(acc=accuracy_score(y, pred), bacc=balanced_accuracy_score(y, pred),
                f1=f1_score(y, pred, zero_division=0), auc=auc)


# ------------------------------------------------------------------ protocols
def run_protocols(name, df, seeds, folds=5):
    """Returns (score rows, fold-geometry rows)."""
    X = df.drop(columns=["y"]).reset_index(drop=True)
    y = df["y"].values
    keys = row_keys(X)                          # feature-level identity
    groups = pd.factorize(keys)[0]

    ded = df.drop_duplicates().reset_index(drop=True)                     # features+label
    Xd, yd = ded.drop(columns=["y"]), ded["y"].values
    keys_d = row_keys(Xd)

    dedf = df.drop_duplicates(subset=[c for c in df.columns if c != "y"]).reset_index(drop=True)
    Xf, yf = dedf.drop(columns=["y"]), dedf["y"].values
    keys_f = row_keys(Xf)

    rows, geom = [], []
    for seed in seeds:
        plans = {
            "P0_naive": (X, y, keys,
                         StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, y)),
            "P1_group": (X, y, keys,
                         StratifiedGroupKFold(folds, shuffle=True, random_state=seed).split(X, y, groups)),
            "P2_dedup": (Xd, yd, keys_d,
                         StratifiedKFold(folds, shuffle=True, random_state=seed).split(Xd, yd)),
            "P2b_dedup_feat": (Xf, yf, keys_f,
                               StratifiedKFold(folds, shuffle=True, random_state=seed).split(Xf, yf)),
        }
        mdls = get_models(seed)
        for proto, (XX, yy, kk, splits) in plans.items():
            for fold, (tr, te) in enumerate(splits):
                overlap = float(np.isin(kk[te], kk[tr]).mean())
                geom.append(dict(dataset=name, protocol=proto, seed=seed, fold=fold,
                                 n_train=len(tr), n_test=len(te),
                                 pos_rate_train=float(yy[tr].mean()),
                                 pos_rate_test=float(yy[te].mean()),
                                 overlap=overlap))
                for mname, model in mdls.items():
                    pipe = Pipeline([("pre", make_preprocessor(XX)), ("clf", clone(model))])
                    pipe.fit(XX.iloc[tr], yy[tr])
                    rows.append(dict(dataset=name, protocol=proto, seed=seed, fold=fold,
                                     model=mname, overlap=overlap, n_test=len(te),
                                     **scores(pipe, XX.iloc[te], yy[te])))
    return rows, geom


def run_smote(name, df, seeds, folds=5):
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    ded = df.drop_duplicates().reset_index(drop=True)
    X, y = ded.drop(columns=["y"]), ded["y"].values
    n_real = len(X)
    rows = []
    for seed in seeds:
        mdls = get_models(seed)
        # (a) correct: resample inside the training folds, test on real rows
        for fold, (tr, te) in enumerate(StratifiedKFold(folds, shuffle=True,
                                                        random_state=seed).split(X, y)):
            for mname, model in mdls.items():
                pipe = ImbPipeline([("pre", make_preprocessor(X)),
                                    ("smote", SMOTE(random_state=seed)),
                                    ("clf", clone(model))])
                pipe.fit(X.iloc[tr], y[tr])
                rows.append(dict(dataset=name, setting="smote_in_fold", seed=seed,
                                 fold=fold, model=mname, n_test=len(te),
                                 **scores(pipe, X.iloc[te], y[te])))
        # resample the whole set once; real rows keep indices 0..n_real-1
        Z = make_preprocessor(X).fit_transform(X)
        Zr, yr = SMOTE(random_state=seed).fit_resample(Z, y)
        is_real = np.arange(len(Zr)) < n_real
        for fold, (tr, te) in enumerate(StratifiedKFold(folds, shuffle=True,
                                                        random_state=seed).split(Zr, yr)):
            te_real = te[is_real[te]]
            for mname, model in mdls.items():
                mdl = clone(model).fit(Zr[tr], yr[tr])
                # (b) what a paper following this procedure reports
                rows.append(dict(dataset=name, setting="smote_pre_resampled", seed=seed,
                                 fold=fold, model=mname, n_test=len(te),
                                 **scores(mdl, Zr[te], yr[te])))
                # (c) same fitted model, scored only on real test rows
                if len(np.unique(yr[te_real])) == 2:
                    rows.append(dict(dataset=name, setting="smote_pre_realtest", seed=seed,
                                     fold=fold, model=mname, n_test=len(te_real),
                                     **scores(mdl, Zr[te_real], yr[te_real])))
    return rows



# ------------------------------------------------------------------ R1.4 capacity ladder
# Same model families at several capacity settings. If capacity is the mechanism, inflation
# should rise with capacity WITHIN a family, not only across families.
def capacity_ladder(seed):
    L = {
        "LR_C0.01":  (LogisticRegression(C=0.01, max_iter=3000), 1),
        "LR_C1":     (LogisticRegression(C=1.0, max_iter=3000), 2),
        "LR_C100":   (LogisticRegression(C=100.0, max_iter=3000), 3),
        "kNN_k25":   (KNeighborsClassifier(n_neighbors=25), 1),
        "kNN_k5":    (KNeighborsClassifier(n_neighbors=5), 2),
        "kNN_k1":    (KNeighborsClassifier(n_neighbors=1), 3),
        "RF_d2":     (RandomForestClassifier(n_estimators=200, max_depth=2,
                                             random_state=seed, n_jobs=-1), 1),
        "RF_d6":     (RandomForestClassifier(n_estimators=200, max_depth=6,
                                             random_state=seed, n_jobs=-1), 2),
        "RF_dNone":  (RandomForestClassifier(n_estimators=200, max_depth=None,
                                             random_state=seed, n_jobs=-1), 3),
    }
    if HAS_XGB:
        L.update({
            "XGB_d2": (XGBClassifier(n_estimators=200, max_depth=2, learning_rate=0.1,
                                     subsample=0.9, eval_metric="logloss",
                                     random_state=seed, n_jobs=-1, verbosity=0), 1),
            "XGB_d4": (XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                     subsample=0.9, eval_metric="logloss",
                                     random_state=seed, n_jobs=-1, verbosity=0), 2),
            "XGB_d8": (XGBClassifier(n_estimators=200, max_depth=8, learning_rate=0.1,
                                     subsample=0.9, eval_metric="logloss",
                                     random_state=seed, n_jobs=-1, verbosity=0), 3),
        })
    return L


def run_capacity_ladder(name, df, seeds, folds=5):
    """P0 vs P1 only; the question is how inflation moves with capacity inside a family."""
    X = df.drop(columns=["y"]).reset_index(drop=True)
    y = df["y"].values
    keys = row_keys(X)
    groups = pd.factorize(keys)[0]
    rows = []
    for seed in seeds:
        L = capacity_ladder(seed)
        plans = {
            "P0_naive": StratifiedKFold(folds, shuffle=True, random_state=seed).split(X, y),
            "P1_group": StratifiedGroupKFold(folds, shuffle=True,
                                             random_state=seed).split(X, y, groups),
        }
        for proto, splits in plans.items():
            for fold, (tr, te) in enumerate(splits):
                overlap = float(np.isin(keys[te], keys[tr]).mean())
                for cfg, (model, rank) in L.items():
                    pipe = Pipeline([("pre", make_preprocessor(X)), ("clf", clone(model))])
                    pipe.fit(X.iloc[tr], y[tr])
                    rows.append(dict(dataset=name, protocol=proto, seed=seed, fold=fold,
                                     config=cfg, family=cfg.split("_")[0],
                                     capacity_rank=rank, overlap=overlap,
                                     **scores(pipe, X.iloc[te], y[te])))
    return rows


# ------------------------------------------------------------------ main
def main(datasets=DATASETS, n_seeds=20, folds=5, outdir="results_v2",
         smote_max_minority=0.40, ladder_seeds=10):
    os.makedirs(outdir, exist_ok=True)
    seeds = list(range(n_seeds))
    audits, cv, geom, sm, lad, log, prof, resid = [], [], [], [], [], {}, {}, {}
    t0 = time.time()
    for name, spec in datasets.items():
        try:
            df, dropped, info = prepare(spec)
        except Exception as e:
            log[name] = f"LOAD FAILED: {e}"; print(f"[{name}] load failed: {e}"); continue
        a = audit(df, "y", eps_list=(0.05, 0.1, 0.25, 0.5))
        a.update(dataset=name, dropped_id_columns=",".join(dropped),
                 positive_label=info["positive"], checksum=frame_checksum(df))
        a.update({f"resid_{k}": v for k, v in
                  residual_overlap_under_dedup(df).items()})
        audits.append(a)
        prof[name] = feature_space_profile(df.drop(columns=["y"]))
        resid[name] = residual_overlap_under_dedup(df)
        print(f"[{name}] n={a['n_rows']} dups={a['exact_dup_rows_feat_label']} "
              f"({a['exact_dup_pct']}%) residual_after_dedup={a['resid_residual_pct']}% "
              f"sha={a['checksum']}")
        r, g = run_protocols(name, df, seeds, folds)
        cv += r; geom += g
        lad += run_capacity_ladder(name, df, seeds[:ladder_seeds], folds)
        if min(df["y"].mean(), 1 - df["y"].mean()) < smote_max_minority:
            sm += run_smote(name, df, seeds, folds)
        log[name] = f"ok ({time.time() - t0:.0f}s)"
        print(f"[{name}] done, {time.time() - t0:.0f}s")

    pd.DataFrame(audits).to_csv(f"{outdir}/table1_audit_v2.csv", index=False)
    pd.DataFrame(cv).to_csv(f"{outdir}/cv_raw_v2.csv", index=False)
    pd.DataFrame(geom).to_csv(f"{outdir}/fold_geometry.csv", index=False)
    if sm:
        pd.DataFrame(sm).to_csv(f"{outdir}/smote_raw_v2.csv", index=False)
    pd.DataFrame(lad).to_csv(f"{outdir}/capacity_ladder.csv", index=False)
    json.dump(prof, open(f"{outdir}/feature_space_profile.json", "w"), indent=1)
    json.dump(resid, open(f"{outdir}/residual_overlap.json", "w"), indent=1)

    import sklearn, scipy, imblearn
    env = dict(python=platform.python_version(), numpy=np.__version__,
               pandas=pd.__version__, scikit_learn=sklearn.__version__,
               scipy=scipy.__version__, imbalanced_learn=imblearn.__version__,
               xgboost=(__import__("xgboost").__version__ if HAS_XGB else None),
               n_seeds=n_seeds, ladder_seeds=ladder_seeds, folds=folds,
               datasets_log=log,
               elapsed_seconds=round(time.time() - t0))
    json.dump(env, open(f"{outdir}/environment.json", "w"), indent=1)
    print(json.dumps(env, indent=1))
    return env


In [ ]:
%%writefile analysis_v2.py
"""Regenerates every table, figure and statistic reported in the paper from the raw
outputs of experiments_v2.main(). Nothing in the manuscript is computed anywhere else.

Usage:  python analysis_v2.py results_v2 analysis_v2
"""
import json, os, sys, warnings
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import pearsonr, spearmanr, wilcoxon
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore")
plt.rcParams.update({"font.size": 7.5, "font.family": "DejaVu Sans",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.linewidth": .7, "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "axes.labelsize": 7.5, "legend.fontsize": 6.5})

MODELS = ["LR", "kNN", "SVM", "RF", "XGB"]
CAP = {"LR": 1, "kNN": 2, "SVM": 3, "RF": 4, "XGB": 5}
COL = dict(zip(MODELS, ["#4C72B0", "#DD8452", "#55A868", "#C44E52", "#8172B3"]))
SHORT = {"heart_kaggle_1025": "HD-1025", "maternal_risk_uci": "MHR",
         "early_diabetes_uci": "ESD", "lung_cancer_kaggle": "LC", "ilpd_uci": "ILPD",
         "ckd_uci": "CKD", "cleveland_uci": "CLEV", "heart_failure_918": "HF-918",
         "stroke_kaggle": "STR"}
METRICS = ["acc", "bacc", "f1", "auc"]
RNG = np.random.default_rng(0)


def boot_ci(v, n=10000):
    bs = np.array([RNG.choice(v, len(v), replace=True).mean() for _ in range(n)])
    return float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def main(indir="results_v2", outdir="analysis_v2"):
    os.makedirs(outdir, exist_ok=True)
    out = {}
    raw = pd.read_csv(f"{indir}/cv_raw_v2.csv")
    geom = pd.read_csv(f"{indir}/fold_geometry.csv")
    aud = pd.read_csv(f"{indir}/table1_audit_v2.csv")
    n_seeds = int(raw.seed.nunique())
    out["n_seeds"] = n_seeds
    out["n_fits"] = int(len(raw))

    # ---------------- seed-level frame -------------------------------------
    sd = raw.groupby(["dataset", "protocol", "model", "seed"], as_index=False)[
            METRICS + ["overlap"]].mean()
    sd.to_csv(f"{outdir}/seed_level_scores.csv", index=False)

    def pair(a, b):
        A = sd[sd.protocol == a].set_index(["dataset", "model", "seed"])
        B = sd[sd.protocol == b].set_index(["dataset", "model", "seed"])
        j = A[["acc", "overlap"]].join(B[["acc"]], rsuffix="_b").reset_index()
        j["infl"] = (j.acc - j.acc_b) * 100
        j["ov"] = j.overlap * 100
        j["cap"] = j.model.map(CAP)
        return j

    d = pair("P0_naive", "P1_group")          # primary
    d2 = pair("P0_naive", "P2_dedup")
    d2b = pair("P0_naive", "P2b_dedup_feat")
    d.to_csv(f"{outdir}/inflation_p0_p1.csv", index=False)

    # ---------------- R1.1 fold geometry + sensitivity ---------------------
    g = geom[["dataset", "protocol", "seed", "fold", "n_train", "n_test",
              "pos_rate_train", "pos_rate_test", "overlap"]]
    fs = g.groupby(["dataset", "protocol", "seed"]).agg(
            n_min=("n_test", "min"), n_max=("n_test", "max"), n_mean=("n_test", "mean"),
            pr_min=("pos_rate_test", "min"), pr_max=("pos_rate_test", "max")).reset_index()
    fs["size_range_pct"] = 100 * (fs.n_max - fs.n_min) / fs.n_mean
    fs["posrate_range_pp"] = 100 * (fs.pr_max - fs.pr_min)
    tbl_geom = fs.groupby(["dataset", "protocol"])[
        ["size_range_pct", "posrate_range_pp"]].mean().round(2).reset_index()
    tbl_geom.to_csv(f"{outdir}/tableA_fold_geometry.csv", index=False)

    # sensitivity 1: fold-size-weighted inflation (weight each fold by its test size)
    w = raw.merge(geom[["dataset", "protocol", "seed", "fold", "n_test"]],
                  on=["dataset", "protocol", "seed", "fold"], suffixes=("", "_g"))
    def wmean(x):
        return np.average(x["acc"], weights=x["n_test"])
    ws = (w.groupby(["dataset", "protocol", "model", "seed"])
            .apply(wmean).rename("acc_w").reset_index())
    A = ws[ws.protocol == "P0_naive"].set_index(["dataset", "model", "seed"])
    B = ws[ws.protocol == "P1_group"].set_index(["dataset", "model", "seed"])
    dw = A[["acc_w"]].join(B[["acc_w"]], rsuffix="_b").reset_index()
    dw["infl_w"] = (dw.acc_w - dw.acc_w_b) * 100

    # sensitivity 2: restrict to seeds whose P1 folds are within 10% size range
    ok = fs[(fs.protocol == "P1_group") & (fs.size_range_pct <= 10)][["dataset", "seed"]]
    d_bal = d.merge(ok, on=["dataset", "seed"])

    sens = []
    for ds in sorted(d.dataset.unique()):
        base = d[d.dataset == ds].groupby("model").infl.mean()
        wrow = dw[dw.dataset == ds].groupby("model").infl_w.mean()
        bal = d_bal[d_bal.dataset == ds].groupby("model").infl.mean() if len(
            d_bal[d_bal.dataset == ds]) else pd.Series(dtype=float)
        p2 = d2[d2.dataset == ds].groupby("model").infl.mean()
        p2b = d2b[d2b.dataset == ds].groupby("model").infl.mean()
        for m in MODELS:
            sens.append(dict(dataset=ds, model=m,
                             infl_p0_p1=round(float(base.get(m, np.nan)), 3),
                             infl_size_weighted=round(float(wrow.get(m, np.nan)), 3),
                             infl_balanced_seeds=(round(float(bal.get(m, np.nan)), 3)
                                                  if len(bal) else None),
                             n_balanced_seeds=int((ok.dataset == ds).sum()),
                             infl_p0_p2=round(float(p2.get(m, np.nan)), 3),
                             infl_p0_p2b=round(float(p2b.get(m, np.nan)), 3)))
    sens = pd.DataFrame(sens)
    sens.to_csv(f"{outdir}/tableB_protocol_sensitivity.csv", index=False)
    agree = sens.dropna(subset=["infl_p0_p1", "infl_p0_p2b"])
    out["sensitivity_max_abs_gap_p1_vs_p2b"] = round(
        float((agree.infl_p0_p1 - agree.infl_p0_p2b).abs().max()), 3)
    out["sensitivity_corr_p1_vs_p2b"] = round(
        float(np.corrcoef(agree.infl_p0_p1, agree.infl_p0_p2b)[0, 1]), 4)
    out["sensitivity_corr_p1_vs_weighted"] = round(float(np.corrcoef(
        sens.infl_p0_p1, sens.infl_size_weighted)[0, 1]), 4)

    # ---------------- R1.2 residual overlap --------------------------------
    ro = geom.groupby(["dataset", "protocol"]).overlap.mean().unstack().round(5)
    ro.to_csv(f"{outdir}/tableC_residual_overlap.csv")
    out["max_overlap_P1"] = float(ro["P1_group"].max())
    out["max_overlap_P2"] = float(ro["P2_dedup"].max())
    out["max_overlap_P2b"] = float(ro["P2b_dedup_feat"].max())

    # ---------------- R1.3 inference ---------------------------------------
    m = d.groupby(["dataset", "model"], as_index=False)[["infl", "ov"]].mean()
    r, p = pearsonr(m.ov, m.infl)
    dl = d.groupby("dataset", as_index=False)[["infl", "ov"]].mean()
    r2, p2v = pearsonr(dl.ov, dl.infl); rs2, ps2 = spearmanr(dl.ov, dl.infl)
    cr = smf.ols("infl ~ ov", data=d).fit(cov_type="cluster",
                                          cov_kwds={"groups": d.dataset})
    try:
        mm = smf.mixedlm("infl ~ ov", d, groups=d["dataset"]).fit(reml=True)
        out["mixedlm_slope"] = dict(beta=round(mm.params["ov"], 4),
            se=round(mm.bse["ov"], 4), p=float(mm.pvalues["ov"]))
    except Exception as e:
        out["mixedlm_slope"] = f"failed: {e}"
    out["pooled_pearson"] = dict(r=round(r, 3), p=float(p))
    out["dataset_level_pearson"] = dict(r=round(r2, 3), p=float(p2v), n=int(len(dl)))
    out["dataset_level_spearman"] = dict(rho=round(rs2, 3), p=float(ps2), n=int(len(dl)))
    out["cluster_robust_slope"] = dict(beta=round(cr.params["ov"], 4),
        se=round(cr.bse["ov"], 4), p=float(cr.pvalues["ov"]),
        ci=[round(x, 4) for x in cr.conf_int().loc["ov"]])

    rows = []
    for (ds, mdl), gg in d.groupby(["dataset", "model"]):
        v = gg.infl.values
        lo, hi = boot_ci(v)
        try:
            pw = float(wilcoxon(v).pvalue) if not np.allclose(v, 0) else 1.0
        except ValueError:
            pw = np.nan
        rows.append(dict(dataset=ds, model=mdl, mean_infl=round(v.mean(), 3),
                         ci_lo=round(lo, 3), ci_hi=round(hi, 3), n_seeds=len(v),
                         wilcoxon_p=pw))
    ci = pd.DataFrame(rows)
    rej, q, _, _ = multipletests(ci.wilcoxon_p.fillna(1).values, alpha=0.05, method="holm")
    ci["p_holm"] = np.round(q, 6); ci["sig_holm"] = rej
    ci.sort_values("mean_infl", ascending=False).to_csv(
        f"{outdir}/tableD_inflation_ci.csv", index=False)
    out["n_sig_holm"] = int(rej.sum()); out["n_tests"] = int(len(ci))
    out["min_attainable_wilcoxon_p"] = float(wilcoxon(np.arange(1, n_seeds + 1)).pvalue)
    out["holm_threshold"] = 0.05 / len(ci)
    out["n_ci_above_zero"] = int((ci.ci_lo > 0).sum())

    # ---------------- R1.4 capacity ----------------------------------------
    try:
        inter = smf.ols("infl ~ ov * cap", data=d).fit(cov_type="cluster",
                                                       cov_kwds={"groups": d.dataset})
        out["interaction_ov_x_cap"] = dict(beta=round(inter.params["ov:cap"], 4),
            se=round(inter.bse["ov:cap"], 4), p=float(inter.pvalues["ov:cap"]),
            ci=[round(x, 4) for x in inter.conf_int().loc["ov:cap"]])
    except Exception as e:
        out["interaction_ov_x_cap"] = f"failed: {e}"
    sl = []
    for mdl in MODELS:
        gg = d[d.model == mdl]
        try:
            f = smf.ols("infl ~ ov", data=gg).fit(cov_type="cluster",
                                                  cov_kwds={"groups": gg.dataset})
            lo, hi = f.conf_int().loc["ov"]
            sl.append(dict(model=mdl, slope=round(f.params["ov"], 4),
                           se=round(f.bse["ov"], 4), ci_lo=round(lo, 4),
                           ci_hi=round(hi, 4), p=float(f.pvalues["ov"])))
        except Exception:
            sl.append(dict(model=mdl, slope=None, se=None, ci_lo=None, ci_hi=None, p=None))
    pd.DataFrame(sl).to_csv(f"{outdir}/tableE_slopes.csv", index=False)
    out["per_model_slopes"] = sl

    lodo = []
    for ds in sorted(d.dataset.unique()):
        sub = d[d.dataset != ds]
        if sub.ov.nunique() < 3:
            continue
        try:
            s_ = {mdl: np.polyfit(sub[sub.model == mdl].ov, sub[sub.model == mdl].infl, 1)[0]
                  for mdl in ["LR", "XGB"]}
        except Exception:
            continue
        lodo.append(dict(excluded=ds, LR=round(s_["LR"], 4), XGB=round(s_["XGB"], 4),
                         ratio=round(s_["XGB"] / s_["LR"], 2) if s_["LR"] else None))
    if lodo:
        lodo = pd.DataFrame(lodo); lodo.to_csv(f"{outdir}/tableF_lodo.csv", index=False)
        rr = lodo.ratio.dropna()
        out["lodo_ratio_range"] = [float(rr.min()), float(rr.max())] if len(rr) else None

    # within-family capacity ladder
    lp = f"{indir}/capacity_ladder.csv"
    if os.path.exists(lp):
        lad = pd.read_csv(lp)
        ls = lad.groupby(["dataset", "protocol", "config", "family", "capacity_rank",
                          "seed"], as_index=False)[["acc"]].mean()
        LA = ls[ls.protocol == "P0_naive"].set_index(
            ["dataset", "config", "family", "capacity_rank", "seed"])
        LB = ls[ls.protocol == "P1_group"].set_index(
            ["dataset", "config", "family", "capacity_rank", "seed"])
        dl2 = LA[["acc"]].join(LB[["acc"]], rsuffix="_b").reset_index()
        dl2["infl"] = (dl2.acc - dl2.acc_b) * 100
        ov = d.groupby("dataset").ov.first()
        dl2["ov"] = dl2.dataset.map(ov)
        fam = []
        for f_, gg in dl2.groupby("family"):
            means = gg.groupby("capacity_rank").infl.mean().round(3).to_dict()
            try:
                ft = smf.ols("infl ~ ov * capacity_rank", data=gg).fit(
                    cov_type="cluster", cov_kwds={"groups": gg.dataset})
                fam.append(dict(family=f_,
                                interaction_beta=round(ft.params["ov:capacity_rank"], 4),
                                p=float(ft.pvalues["ov:capacity_rank"]),
                                mean_infl_by_rank=means))
            except Exception as e:
                fam.append(dict(family=f_, interaction_beta=None, p=None,
                                mean_infl_by_rank=means, note=str(e)[:80]))
        out["within_family_capacity"] = fam
        dl2.groupby(["family", "config", "capacity_rank"]).infl.mean().round(3)\
           .reset_index().to_csv(f"{outdir}/tableG_capacity_ladder.csv", index=False)

    # ---------------- R1.5 oversampling ------------------------------------
    sp = f"{indir}/smote_raw_v2.csv"
    if os.path.exists(sp):
        sm = pd.read_csv(sp)
        ss = sm.groupby(["dataset", "setting", "model", "seed"], as_index=False)[METRICS].mean()
        piv = ss.pivot_table(index=["dataset", "model", "seed"], columns="setting",
                             values=METRICS)
        res = []
        for met in METRICS:
            try:
                leak = (piv[(met, "smote_pre_realtest")] - piv[(met, "smote_in_fold")]) * 100
                comp = (piv[(met, "smote_pre_resampled")] - piv[(met, "smote_pre_realtest")]) * 100
                tot = (piv[(met, "smote_pre_resampled")] - piv[(met, "smote_in_fold")]) * 100
            except KeyError:
                continue
            t = pd.DataFrame(dict(leakage=leak, composition=comp, total=tot)).reset_index()
            t["metric"] = met
            res.append(t)
        if res:
            dec = pd.concat(res)
            dec.to_csv(f"{outdir}/smote_decomposition_raw.csv", index=False)
            summ = dec.groupby(["metric", "dataset", "model"])[
                ["leakage", "composition", "total"]].mean().round(3).reset_index()
            summ.to_csv(f"{outdir}/tableH_smote_decomposition.csv", index=False)
            out["smote_decomposition"] = {
                met: dict(leakage_mean=round(float(gg.leakage.mean()), 2),
                          leakage_max=round(float(gg.leakage.max()), 2),
                          composition_mean=round(float(gg.composition.mean()), 2),
                          composition_max=round(float(gg.composition.max()), 2))
                for met, gg in dec.groupby("metric")}

    # ---------------- figures ----------------------------------------------
    mm2 = d.groupby(["dataset", "model"], as_index=False)[["infl", "ov"]].mean()
    fig, ax = plt.subplots(figsize=(3.4, 2.35))
    for mdl in MODELS:
        gg = mm2[mm2.model == mdl]
        k, b = np.polyfit(gg.ov, gg.infl, 1); rr, _ = pearsonr(gg.ov, gg.infl)
        ax.plot([0, 100], [b, k * 100 + b], color=COL[mdl], lw=1.1, alpha=.85)
        ax.scatter(gg.ov, gg.infl, color=COL[mdl], s=16, zorder=3,
                   label=f"{mdl}: slope {k:.3f}, r={rr:.2f}")
    ax.axhline(0, color="grey", lw=.5); ax.set_xlim(-4, 104)
    ax.set_xlabel("Train–test overlap (% of test rows)")
    ax.set_ylabel("Accuracy inflation (pp)")
    ax.legend(frameon=False, loc="upper left", handletextpad=.4, borderpad=.1,
              labelspacing=.25)
    fig.tight_layout(pad=.3); fig.savefig(f"{outdir}/fig1.png", dpi=500); plt.close(fig)

    try:
        mm2["short"] = mm2.dataset.map(lambda x: SHORT.get(x, x))
        order = mm2.groupby("short").ov.first().sort_values(ascending=False).index
        pv = mm2.pivot(index="short", columns="model", values="infl").loc[order, MODELS]
        ovs = mm2.groupby("short").ov.first().loc[order]
        fig, ax = plt.subplots(figsize=(3.4, 2.2))
        x = np.arange(len(order))
        for i, mdl in enumerate(MODELS):
            ax.bar(x + (i - 2) * .165, pv[mdl], .165, color=COL[mdl], label=mdl, lw=0)
        ax.axhline(0, color="grey", lw=.5); ax.set_xticks(x)
        ax.set_xticklabels([f"{s}\n{o:.0f}%" for s, o in zip(order, ovs)], fontsize=6.2)
        ax.set_ylabel("Accuracy inflation (pp)")
        ax.legend(frameon=False, ncol=5, columnspacing=.8, handlelength=.9,
                  handletextpad=.3, loc="upper right")
        fig.tight_layout(pad=.3); fig.savefig(f"{outdir}/fig2.png", dpi=500)
        plt.close(fig)
    except Exception as e:
        out["fig2_error"] = str(e)[:120]

    if "smote_decomposition" in out:
      try:
          fig, axes = plt.subplots(1, 2, figsize=(3.4, 2.0))
          for ax, met, lab in zip(axes, ["bacc", "f1"], ["Balanced acc.", "F1"]):
              gg = dec[dec.metric == met].groupby("dataset")[
                  ["leakage", "composition"]].mean()
              gg.index = [SHORT.get(i, i) for i in gg.index]
              gg = gg.loc[gg.sum(axis=1).sort_values().index]
              xx = np.arange(len(gg))
              ax.bar(xx, gg.leakage, .5, label="leakage", color="#C44E52", lw=0)
              ax.bar(xx, gg.composition, .5, bottom=gg.leakage, label="test composition",
                     color="#4C72B0", lw=0)
              ax.axhline(0, color="grey", lw=.5); ax.set_xticks(xx)
              ax.set_xticklabels(gg.index, rotation=55, ha="right", fontsize=6)
              ax.set_ylabel(f"{lab} inflation (pp)", fontsize=6.8)
              ax.tick_params(axis="y", labelsize=6.5)
          axes[0].legend(frameon=False, fontsize=5.8, loc="upper left")
          fig.tight_layout(pad=.3, w_pad=1.2)
          fig.savefig(f"{outdir}/fig3.png", dpi=500); plt.close(fig)
      except Exception as e:
          out["fig3_error"] = str(e)[:120]

    json.dump(out, open(f"{outdir}/headline_v2.json", "w"), indent=1, default=str)
    print(json.dumps(out, indent=1, default=str))
    return out


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else "results_v2",
         sys.argv[2] if len(sys.argv) > 2 else "analysis_v2")


## 1) Quick test — 1 seed
Confirm every dataset loads. No line should say `LOAD FAILED`.

In [ ]:
import importlib, experiments_v2 as E
importlib.reload(E)
E.main(n_seeds=1, ladder_seeds=1, outdir='results_quick')

## 2) Full run — 20 seeds, capacity ladder at 10

20 seeds, not 10: with 45 paired tests the smallest attainable Wilcoxon p at 10 seeds is 0.00195, above the Holm threshold of 0.00111, so no cell could reach significance however large the effect. At 20 seeds the floor is about 2e-6.

In [ ]:
import importlib, experiments_v2 as E
importlib.reload(E)
E.main(n_seeds=20, ladder_seeds=10, outdir='results_v2')

## 3) Analysis — every table, figure and statistic in the paper

In [ ]:
import importlib, analysis_v2 as A
importlib.reload(A)
A.main('results_v2', 'analysis_v2')

## 4) Package

In [ ]:
import shutil, os
os.makedirs('out', exist_ok=True)
shutil.copytree('results_v2','out/results_v2', dirs_exist_ok=True)
shutil.copytree('analysis_v2','out/analysis_v2', dirs_exist_ok=True)
shutil.make_archive('/kaggle/working/results_v2','zip','out')
print('Done -> Output panel -> results_v2.zip')